# Data preparation

**Source:** [UCI Beijing Multi-Site Air Quality dataset](https://archive.ics.uci.edu/dataset/501/beijing+multi+site+air+quality+data) (Chen, 2017, CC BY 4.0).
It has 12 CSV files, one per monitoring station, with hourly readings from 1 March 2013 to 28 February 2017.

The raw files are not included in this repo because of their size. Download them from the link above into a folder called `raw/` to run this notebook.
The output of this notebook, `data/beijing_hourly_city.csv` (about 3 MB), is included so the other notebooks run straight away.

In [ ]:
import pandas as pd
import glob

files = sorted(glob.glob("../raw/PRSA_Data_*.csv"))
print(len(files), "station files found")

data = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
data["datetime"] = pd.to_datetime(data[["year", "month", "day", "hour"]])
data = data.sort_values(["station", "datetime"])
print(data.shape)
data.isnull().sum()

In [ ]:
# fill missing values inside each station by linear interpolation (time order)
num_cols = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM"]
data[num_cols] = data.groupby("station")[num_cols].transform(lambda s: s.interpolate(limit_direction="both"))

# wind direction is a category, so carry the last known value forward
data["wd"] = data.groupby("station")["wd"].transform(lambda s: s.ffill().bfill())
data.isnull().sum().sum()

In [ ]:
# city-wide value for each hour = average of the 12 stations
city = data.groupby("datetime")[num_cols].mean().round(2)

# most common wind direction across the stations for that hour
city["wd"] = data.groupby("datetime")["wd"].agg(lambda s: s.mode().iloc[0])

# daily PM2.5 for each station (used for station comparison)
station_daily = data.set_index("datetime").groupby("station")["PM2.5"].resample("D").mean().round(2).unstack(0)

city.to_csv("../data/beijing_hourly_city.csv")
station_daily.to_csv("../data/station_daily_pm25.csv")
print(city.shape, station_daily.shape)